In [79]:
'''basics'''
import pandas as pd
import transformers
print('transformers: {}'.format(transformers.__version__))

'''visualisations'''
import plotly.express as px

transformers: 4.17.0


In [80]:
df= pd.read_csv('../data/raw/osdg-community-dataset-v2022-04-01.csv',sep='\t')

print(df.shape)
df.head(5)

(32327, 7)


,doi,text_id,text,sdg,labels_negative,labels_positive,agreement
0,10.6027/9789289342698-7-en,00021941702cd84171ff33962197ca1f,"From a gender perspective, Paulgaard points ou...",5,1,8,0.777778
1,10.18356/eca72908-en,00028349a7f9b2485ff344ae44ccfd6b,Labour legislation regulates maximum working h...,11,2,1,0.333333
2,10.1787/9789264289062-4-en,0004eb64f96e1620cd852603d9cbe4d4,The average figure also masks large difference...,3,1,8,0.777778
3,10.1787/5k9b7bn5qzvd-en,0006a887475ccfa5a7f5f51d4ac83d02,The extent to which they are akin to corruptio...,3,1,2,0.333333
4,10.1787/9789264258211-6-en,0006d6e7593776abbdf4a6f985ea6d95,A region reporting a higher rate will not earn...,3,2,2,0.000000


In [81]:
print('average text length: ', df.text.str.split().str.len().mean().round(2))
print('stdev text length: ', df.text.str.split().str.len().std().round(2))
print('max text length: ', df.text.str.split().str.len().max().round(2))

if df.text.str.split().str.len().mean() < 256:
    print("suitable for standard transformer models!")

average text length:  89.79
stdev text length:  29.31
max text length:  226
suitable for standard transformer models!


In [82]:
'''plot sdg distribution'''
fig = px.histogram(df, x="sdg",  nbins=30, title="SDG Distribution")
fig.update_layout(
    xaxis = dict(
        tickmode = 'linear',
        tick0 = 1,
        dtick = 1
    )
)
fig.show()

print("not evenly distributed - check when training multilabel/mutliclass classifier")

print("interestingly we see that only 15 SDGs are included. 16 and 17 seem to be missing")

not evenly distributed - check when training multilabel/mutliclass classifier
interestingly we see that only 15 SDGs are included. 16 and 17 seem to be missing


In [83]:
'''plot share of positive/negative labels per sdgs to visualise agreement of annotators'''
count_sdg = df[['sdg', 'labels_negative', 'labels_positive']].groupby('sdg', as_index=False).sum()
fig = px.bar(count_sdg, x="sdg", y=["labels_negative", "labels_positive"], title="Annotation Distribution")
fig.update_layout(
    xaxis = dict(
        tickmode = 'linear',
        tick0 = 1,
        dtick = 1
    )
)
fig.show()

print("agreement looks are okayish...")


agreement looks are okayish...


In [89]:
# keeping only the texts whose suggested sdg labels is accepted and the agreement score is at least .7
# We want to have pretty high confidence in our labels and thus drop almost 50% of the training data. Quality > Quantity
print('Shape before:', df.shape)
df = df.query('agreement >= .7 and labels_positive > labels_negative').copy()
print('Shape after :', df.shape)
display(df.head())

Shape before: (16660, 7)
Shape after : (16660, 7)


,doi,text_id,text,sdg,labels_negative,labels_positive,agreement
0,10.6027/9789289342698-7-en,00021941702cd84171ff33962197ca1f,"From a gender perspective, Paulgaard points ou...",5,1,8,0.777778
2,10.1787/9789264289062-4-en,0004eb64f96e1620cd852603d9cbe4d4,The average figure also masks large difference...,3,1,8,0.777778
7,10.1787/9789264117563-8-en,000bfb17e9f3a00d4515ab59c5c487e7,The Israel Oceanographic and Limnological Rese...,6,0,3,1.000000
8,10.18356/805b1ae4-en,001180f5dd9a821e651ed51e30d0cf8c,Previous chapters have discussed ways to make ...,2,0,3,1.000000
11,10.1787/9789264310278-en,001f1aee4013cb098da17a979c38bc57,Prescription rates appear to be higher where l...,8,0,3,1.000000


In [85]:
#aggregate 
df_lambda = df.groupby('sdg', as_index = False).agg(count = ('text_id', 'count'))
df_lambda['share'] = df_lambda['count'].divide(df_lambda['count'].sum()).multiply(100)
print('Shape:', df_lambda.shape)
display(df_lambda.head())

Shape: (15, 3)


,sdg,count,share
0,1,1092,6.554622
1,2,761,4.567827
2,3,1787,10.726291
3,4,2295,13.775510
4,5,2262,13.577431


In [86]:
fig = px.bar(
    data_frame = df_lambda,
    x = 'sdg',
    y = 'count',
    custom_data = ['share'],
    labels = {
        'sdg': 'SDG',
        'count': 'Count'
    },
    color_discrete_sequence = ['#1f77b4'],
    title = 'Figure 2. Distribution of Texts (Agreement >.6) over SDGs'
)

fig.update_traces(hovertemplate = 'SDG %{x}<br>Count: %{y}<br>Share: %{customdata:.2f}%')
fig.update_layout(xaxis = {'type': 'category'})
fig.show()